In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, time
import pytz
import re

# ==========================================================
# LOAD DATA
# ==========================================================
def load_data():
    play_df = pd.read_csv("Play Store Data.csv")
    review_df = pd.read_csv("User Reviews.csv")

    play_df.columns = play_df.columns.str.strip()
    review_df.columns = review_df.columns.str.strip()

    play_df["Installs"] = play_df["Installs"].astype(str).str.replace("[+,]", "", regex=True)
    play_df["Installs"] = pd.to_numeric(play_df["Installs"], errors="coerce")
    play_df["Reviews"] = pd.to_numeric(play_df["Reviews"], errors="coerce")
    play_df["Rating"] = pd.to_numeric(play_df["Rating"], errors="coerce")
    play_df["Size_MB"] = play_df["Size"].astype(str).str.replace("M","").replace("Varies with device", np.nan)
    play_df["Size_MB"] = pd.to_numeric(play_df["Size_MB"], errors="coerce")
    play_df["Last Updated"] = pd.to_datetime(play_df["Last Updated"], errors="coerce")

    def clean_android(ver):
        if pd.isna(ver): return np.nan
        match = re.search(r"\d+(\.\d+)?", str(ver))
        return float(match.group()) if match else np.nan
    play_df["Android_Ver_Num"] = play_df["Android Ver"].apply(clean_android)

    review_df["Sentiment_Subjectivity"] = pd.to_numeric(review_df["Sentiment_Subjectivity"], errors="coerce")
    review_avg = review_df.groupby("App")["Sentiment_Subjectivity"].mean().reset_index()
    merged_df = play_df.merge(review_avg, on="App", how="left")

    return merged_df

df = load_data()
print("✅ Data Loaded: Play Store + User Reviews merged successfully")

# ==========================================================
# CURRENT IST TIME
# ==========================================================
def get_ist_time():
    ist = pytz.timezone("Asia/Kolkata")
    return datetime.now(ist).time()

current_time = get_ist_time()
print(f"🕒 Current IST Time: {current_time}")

# ==========================================================
# TASK 1: GROUPED BAR CHART (3 PM – 5 PM)
# ==========================================================
def task1():
    if not (time(15,0) <= current_time <= time(17,0)):
        print("⏰ Task 1 visible only between 3 PM – 5 PM IST")
        return

    temp = df[(df["Rating"] >= 4.0) & (df["Size_MB"] >= 10) & (df["Last Updated"].dt.month == 1)]
    top10 = temp.groupby("Category").agg({"Installs":"sum","Rating":"mean","Reviews":"sum"}).sort_values("Installs", ascending=False).head(10)
    
    fig = go.Figure(data=[
        go.Bar(name="Total Reviews", x=top10.index, y=top10["Reviews"]),
        go.Bar(name="Avg Rating", x=top10.index, y=top10["Rating"], yaxis="y2")
    ])
    
    fig.update_layout(
        title="Task 1: Reviews vs Avg Rating by Category",
        yaxis=dict(title="Total Reviews"),
        yaxis2=dict(title="Avg Rating", overlaying="y", side="right"),
        barmode='group'
    )
    fig.show()

# ==========================================================
# TASK 2: DUAL AXIS CHART (1 PM – 2 PM)
# ==========================================================
def task2():
    if not (time(13,0) <= current_time <= time(14,0)):
        print("⏰ Task 2 visible only between 1 PM – 2 PM IST")
        return

    temp = df[(df["Installs"] >= 10000) & (df["Android_Ver_Num"] > 4.0) & 
              (df["Size_MB"] > 15) & (df["Content Rating"] == "Everyone") & (df["App"].str.len() <= 30)]

    np.random.seed(42)
    temp["Price"] = np.random.choice([0,2,5,10], size=len(temp))
    temp["Revenue"] = temp["Installs"] * temp["Price"]
    temp = temp[temp["Revenue"] >= 10000]
    temp["Type"] = temp["Price"].apply(lambda x: "Free" if x==0 else "Paid")

    top3 = temp["Category"].value_counts().nlargest(3).index
    temp = temp[temp["Category"].isin(top3)]
    agg = temp.groupby(["Category","Type"]).agg(Avg_Installs=("Installs","mean"), Avg_Revenue=("Revenue","mean")).reset_index()

    fig = go.Figure()
    for cat in top3:
        subset = agg[agg["Category"]==cat]
        fig.add_trace(go.Bar(name=f"{cat} - Free", x=[cat], y=subset[subset["Type"]=="Free"]["Avg_Installs"]))
        fig.add_trace(go.Bar(name=f"{cat} - Paid", x=[cat], y=subset[subset["Type"]=="Paid"]["Avg_Installs"]))
        fig.add_trace(go.Scatter(name=f"{cat} Revenue", x=[cat], y=[subset["Avg_Revenue"].mean()], mode="markers+lines", yaxis="y2"))

    fig.update_layout(
        title="Task 2: Avg Installs & Revenue by Category & Type",
        yaxis=dict(title="Avg Installs"),
        yaxis2=dict(title="Avg Revenue", overlaying="y", side="right"),
        barmode='group'
    )
    fig.show()

# ==========================================================
# TASK 3: CHOROPLETH MAP (6 PM – 8 PM)
# ==========================================================
def task3():
    if not (time(18,0) <= current_time <= time(20,0)):
        print("⏰ Task 3 visible only between 6 PM – 8 PM IST")
        return

    temp = df[~df["Category"].str.startswith(("A","C","G","S"))]
    top5 = temp.groupby("Category")["Installs"].sum().nlargest(5).index
    temp = temp[temp["Category"].isin(top5)]

    countries = ["India","United States","Germany","Brazil","Japan"]
    temp["Country"] = temp["App"].apply(lambda x: countries[hash(x)%5])
    agg = temp.groupby(["Country","Category"])["Installs"].sum().reset_index()

    fig = px.choropleth(
        agg,
        locations="Country",
        locationmode="country names",
        color="Installs",
        hover_name="Category",
        title="Task 3: Global Installs by Category"
    )
    fig.show()

# ==========================================================
# TASK 4: STACKED AREA CHART (4 PM – 6 PM)
# ==========================================================
def task4():
    if not (time(16,0) <= current_time <= time(18,0)):
        print("⏰ Task 4 visible only between 4 PM – 6 PM IST")
        return

    temp = df[(df["Rating"] >= 4.2) & (df["Reviews"] > 1000) & (df["Size_MB"].between(20,80)) &
              (~df["App"].str.contains(r"\d")) & (df["Category"].str.startswith(("T","P")))]
    temp["Month"] = temp["Last Updated"].dt.to_period("M").astype(str)
    pivot = temp.groupby(["Month","Category"])["Installs"].sum().unstack().fillna(0).cumsum()

    fig = go.Figure()
    for col in pivot.columns:
        fig.add_trace(go.Scatter(x=pivot.index, y=pivot[col], stackgroup='one', name=col))
    fig.update_layout(title="Task 4: Cumulative Installs by Month & Category")
    fig.show()

# ==========================================================
# TASK 5: SCATTER PLOT (5 PM – 7 PM)
# ==========================================================
def task5():
    if not (time(17,0) <= current_time <= time(19,0)):
        print("⏰ Task 5 visible only between 5 PM – 7 PM IST")
        return

    allowed = ["GAME","BEAUTY","BUSINESS","COMICS","COMMUNICATION","DATING","ENTERTAINMENT","SOCIAL","EVENTS"]
    temp = df[(df["Rating"] > 3.5) & (df["Reviews"] > 500) & (df["Installs"] > 50000) & 
              (df["Sentiment_Subjectivity"] > 0.5) & (~df["App"].str.contains("S", case=False)) & 
              (df["Category"].str.upper().isin(allowed))]

    temp["Color"] = temp["Category"].apply(lambda x: "Game" if x=="GAME" else "Other")
    fig = px.scatter(temp, x="Size_MB", y="Rating", size="Installs", color="Color",
                     color_discrete_map={"Game":"pink","Other":"blue"},
                     title="Task 5: App Size vs Rating")
    fig.show()

# ==========================================================
# TASK 6: TIME SERIES LINE CHART (6 PM – 9 PM)
# ==========================================================
def task6():
    if not (time(18,0) <= current_time <= time(21,0)):
        print("⏰ Task 6 visible only between 6 PM – 9 PM IST")
        return

    temp = df[(df["Reviews"] > 500) & (~df["App"].str.lower().str.startswith(("x","y","z"))) &
              (~df["App"].str.contains("S", case=False)) & (df["Category"].str.startswith(("E","C","B")))]
    temp["Month"] = temp["Last Updated"].dt.to_period("M").astype(str)
    pivot = temp.groupby(["Month","Category"])["Installs"].sum().unstack().fillna(0)

    fig = go.Figure()
    for col in pivot.columns:
        fig.add_trace(go.Scatter(x=pivot.index, y=pivot[col], mode="lines+markers", name=col))
    fig.update_layout(title="Task 6: Time Series of Installs by Category")
    fig.show()

# ==========================================================
# RUN TASKS
# ==========================================================
task1()
task2()
task3()
task4()
task5()
task6()


✅ Data Loaded: Play Store + User Reviews merged successfully
🕒 Current IST Time: 21:01:55.654600
⏰ Task 1 visible only between 3 PM – 5 PM IST
⏰ Task 2 visible only between 1 PM – 2 PM IST
⏰ Task 3 visible only between 6 PM – 8 PM IST
⏰ Task 4 visible only between 4 PM – 6 PM IST
⏰ Task 5 visible only between 5 PM – 7 PM IST
⏰ Task 6 visible only between 6 PM – 9 PM IST
